# Clustering with the cleaned text and the multilabel stance scores
Part A: does cleaned text and/or the multilabel scores give better comment clusters?  
Part B: do day-level discourse regimes line up with EM-DAT events?

In [7]:
import numpy as np, pandas as pd, pyarrow.parquet as pq
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

SCORED = "reddit_climate_stance_scored.parquet"   # new multilabel file (p_support, p_skeptic), in scripts/
EMDAT = "../data/emdatclimatedata.xlsx"
COMMUNITIES = ["climatechange", "climate", "energy", "Futurology", "conspiracy", "changemyview"]
START, END = "2023-11-19", "2026-09-01"           # same window as 09c / ARM
MIN_WORDS = 20                                     # same as 09c
GATE = 0.5                                         # a comment "takes a stance" if max(p_support, p_skeptic) >= GATE
N_PER_COMM = 15000                                  # Part A: comments per community
K = 12                                             # same k as 09c
W = 0.5                                            # weight of the stance probabilities when appended to embeddings
EMBEDDER = "minilm"                                # "minilm" or "lsa" (TF-IDF + SVD, no model download)
LAG = 3                                            # an event affects days start .. start+LAG
SEED = 42

## Load (one streaming pass)

In [8]:
f = pq.ParquetFile(SCORED)
assert {"p_support", "p_skeptic"} <= set(f.schema_arrow.names), "no multilabel columns in this file"
cols = ["subreddit", "created_time", "self_text", "clean_text", "p_support", "p_skeptic", "comment_word_count"]
rng = np.random.default_rng(SEED)
sample, daily = [], []
for b in f.iter_batches(batch_size=100_000, columns=cols):
    d = b.to_pandas()
    d = d[d.subreddit.isin(COMMUNITIES) & (d.created_time >= START) & (d.created_time < END) & (d.comment_word_count >= MIN_WORDS)]
    daily.append(d[["created_time", "p_support", "p_skeptic"]])
    sample.append(d.sample(frac=0.1, random_state=int(rng.integers(1e9))))
daily, sample = pd.concat(daily), pd.concat(sample)
print(f"{len(daily):,} comments in scope, {len(sample):,} sampled")

623,186 comments in scope, 62,320 sampled


## Part A: comment clusters

In [9]:
n = min(N_PER_COMM, sample.subreddit.value_counts().min())
s = sample.groupby("subreddit", group_keys=False).sample(n, random_state=SEED).reset_index(drop=True)
P = s[["p_support", "p_skeptic"]].to_numpy()
bearing = P.max(axis=1) >= GATE
print(f"{len(s):,} comments, {bearing.mean():.1%} take a stance")
s.assign(bearing=bearing).groupby("subreddit").bearing.mean().round(3)

38,946 comments, 22.1% take a stance


subreddit
Futurology       0.181
changemyview     0.067
climate          0.349
climatechange    0.366
conspiracy       0.080
energy           0.281
Name: bearing, dtype: float64

In [10]:
def embed(texts):
    if EMBEDDER == "minilm":
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer("all-MiniLM-L6-v2").encode(list(texts), batch_size=128, normalize_embeddings=True)
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize
    X = TfidfVectorizer(max_features=30000, min_df=3, stop_words="english", sublinear_tf=True).fit_transform(texts)
    return normalize(TruncatedSVD(100, random_state=SEED).fit_transform(X))

E_raw, E_clean = embed(s.self_text), embed(s.clean_text)

c:\Users\bhatr\Documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\bhatr\Documents\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhatr\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activa

In [11]:
def km(X, seed=SEED):
    return KMeans(K, n_init=3, random_state=seed).fit_predict(X)

def stability(X):
    a = km(X)
    return a, np.mean([adjusted_rand_score(a, km(X, SEED + i)) for i in (1, 2)])

def eta2(v, y):   # share of the variance of v explained by the clusters
    within = sum(((v[y == c] - v[y == c].mean()) ** 2).sum() for c in np.unique(y))
    return 1 - within / ((v - v.mean()) ** 2).sum()

def row(name, y, m, stab):   # evaluate labels y on rows m, always in the cleaned-text embedding space
    return dict(variant=name, rows=int(m.sum()), stability_ARI=round(stab, 3),
                silhouette=round(silhouette_score(E_clean[m], y, metric="cosine", sample_size=min(10000, int(m.sum())), random_state=SEED), 3),
                eta2_support=round(eta2(P[m, 0], y), 3), eta2_skeptic=round(eta2(P[m, 1], y), 3))

allrows = np.ones(len(s), bool)
yA, sA = stability(E_raw)
yB, sB = stability(E_clean)
yC, sC = stability(np.hstack([E_clean, W * P]))
yD, sD = stability(E_clean[bearing])

pd.DataFrame([
    row("A raw text", yA, allrows, sA),
    row("B cleaned text", yB, allrows, sB),
    row("C cleaned + stance probs", yC, allrows, sC),
    row("A raw text (stance-bearing rows)", yA[bearing], bearing, sA),
    row("B cleaned text (stance-bearing rows)", yB[bearing], bearing, sB),
    row("D cleaned, stance-bearing only", yD, bearing, sD),
])

,variant,rows,stability_ARI,silhouette,eta2_support,eta2_skeptic
0,A raw text,38946,0.738,0.031,0.230,0.103
1,B cleaned text,38946,0.786,0.030,0.216,0.100
2,C cleaned + stance probs,38946,0.577,0.009,0.848,0.770
3,A raw text (stance-bearing rows),8596,0.738,0.062,0.133,0.143
4,B cleaned text (stance-bearing rows),8596,0.786,0.060,0.133,0.142
5,"D cleaned, stance-bearing only",8596,0.762,0.046,0.179,0.188


## Part B: day-level regimes vs EM-DAT

In [12]:
daily["date"] = daily.created_time.dt.floor("D")
daily["sup"], daily["skp"] = daily.p_support >= GATE, daily.p_skeptic >= GATE
daily["bearing"] = daily.sup | daily.skp
g = daily.groupby("date")
day = pd.DataFrame({"log_volume": np.log(g.size()), "mean_support": g.p_support.mean(), "mean_skeptic": g.p_skeptic.mean(),
                    "share_bearing": g.bearing.mean(), "support_share": g.sup.sum() / g.bearing.sum().replace(0, np.nan)})
day = day.reindex(pd.date_range(day.index.min(), day.index.max()))

x = day - day.rolling(29, center=True, min_periods=15).mean()        # drop slow trends and seasonality
x = (x - x.groupby(x.index.dayofweek).transform("mean")).dropna()    # drop the weekday pattern

Z = StandardScaler().fit_transform(x)
sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)) for k in range(2, 7)}
k = max(sil, key=sil.get)
x["regime"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)
print({k_: round(v, 3) for k_, v in sil.items()}, "-> k =", k)
x.groupby("regime").mean().round(3).assign(days=x.regime.value_counts())

{2: 0.305, 3: 0.281, 4: 0.237, 5: 0.227, 6: 0.228} -> k = 2


,log_volume,mean_support,mean_skeptic,share_bearing,support_share,days
regime,,,,,,
0,0.057,-0.029,-0.011,-0.038,-0.003,552
1,-0.073,0.037,0.014,0.049,0.004,429


In [14]:
ev = pd.read_excel(EMDAT).dropna(subset=["Start Month", "Start Day"])
ev["date"] = pd.to_datetime(dict(year=ev["Start Year"], month=ev["Start Month"], day=ev["Start Day"]), errors="coerce")
ev = ev[ev.date.between(day.index.min(), day.index.max())]
hit = pd.Series(0.0, index=day.index)
for d in ev.date:
    hit.loc[d : d + pd.Timedelta(days=LAG)] = 1
print(f"{len(ev)} events, {hit.mean():.1%} of calendar days fall within {LAG} days of one")

65 events, 20.2% of calendar days fall within 3 days of one


In [15]:
# Null: slide the event calendar by a random number of days (keeps its clumping, breaks any link to the discourse)
ok = day.index.isin(x.index)
h = hit.to_numpy()
H = np.stack([np.roll(h, s_)[ok] for s_ in np.random.default_rng(SEED).integers(30, len(h) - 30, 5000)])
h_ok = h[ok]
pval = lambda obs, null: round((1 + (np.abs(null) >= abs(obs)).sum()) / (1 + len(null)), 4)

res = []
for r in sorted(x.regime.unique()):
    m = (x.regime == r).to_numpy()
    obs, null = h_ok[m].mean() - h_ok.mean(), H[:, m].mean(1) - H.mean(1)
    res.append(dict(what=f"regime {r}", days=int(m.sum()), event_day_share=round(h_ok[m].mean(), 3), diff_vs_all=round(obs, 3), p=pval(obs, null)))

Hc = (H - H.mean(1, keepdims=True)) / H.std(1, keepdims=True)
for c in day.columns:
    v = rankdata(x[c]); v = (v - v.mean()) / v.std()
    obs, null = np.corrcoef(v, h_ok)[0, 1], Hc @ v / len(v)
    res.append(dict(what=f"spearman: {c}", diff_vs_all=round(obs, 3), p=pval(obs, null)))
pd.DataFrame(res)

,what,days,event_day_share,diff_vs_all,p
0,regime 0,552.0,0.167,-0.028,0.0158
1,regime 1,429.0,0.231,0.036,0.0158
2,spearman: log_volume,NaN,NaN,0.022,0.5299
3,spearman: mean_support,NaN,NaN,0.067,0.0554
4,spearman: mean_skeptic,NaN,NaN,0.052,0.1318
5,spearman: share_bearing,NaN,NaN,0.076,0.0326
6,spearman: support_share,NaN,NaN,-0.000,0.9992
